# TerraCast · Ensemble Yield Prediction + MLflow 3

Corn / Soybeans split model with:
- State / County fixed effects
- Ensemble: LightGBM + Random Forest (simple average)
- MLflow 3 experiment tracking (LoggedModel, Dataset linking)
- 5-Fold CV anomaly detection

**Run in Databricks** (Spark available)

In [9]:
%pip install -U 'mlflow>=3.1' lightgbm geopandas scikit-learn shap pygris


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import KFold, train_test_split

import mlflow
import mlflow.lightgbm
import mlflow.sklearn
from mlflow.entities import Dataset

# Spark session in Databricks; None locally
try:
    spark = globals().get("spark", None)
except NameError:
    spark = None

print("MLflow version:", mlflow.__version__)

MLflow version: 3.13.0


## Config

In [11]:
# --- Edit these ---
MLFLOW_EXPERIMENT_ID = "2445807330421530"   # Databricks MLflow experiment ID
MERGED_TABLE         = "workspace.default.merged"  # Spark table (Databricks)
CSV_PATH             = "/Users/othree/TerraCast/TerraCast/models/databricks_hackathon/Merged.csv"                 # local CSV fallback, e.g. Path("/path/to/merged.csv")

# Year-based train/test split (prevents time leakage)
TEST_START_YEAR = 2021  # years >= 2021 are test, < 2021 are train

# Region map (state → region) — based on USDA regional classification
# Northeast merged into adjacent regions to avoid data sparsity:
#   NY, PA, VT, CT, MA, ME, NH, RI, NJ → Midwest (northern climate)
#   MD, DE → South (mid-Atlantic climate)
REGION_MAP = {
    "Midwest": {
        "IA", "IL", "IN", "MI", "MN", "MO", "OH", "WI",   # Corn Belt core
        "NY", "PA", "VT", "CT", "MA", "ME", "NH", "RI", "NJ",  # Northeast (northern)
    },
    "Plains": {
        "CO", "KS", "MT", "ND", "NE", "OK", "SD", "WY",   # Great Plains
    },
    "South": {
        "AL", "AR", "GA", "KY", "LA", "MS", "NC", "SC", "TN", "TX", "VA", "WV",
        "MD", "DE",  # mid-Atlantic
    },
}

MIN_TRAIN_ROWS = 200  # skip crop×region combinations below this threshold

# LightGBM hyperparameters
LGBM_PARAMS = dict(
    n_estimators      = 1000,
    learning_rate     = 0.05,
    num_leaves        = 63,
    min_child_samples = 20,
    subsample         = 0.8,
    colsample_bytree  = 0.6,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    random_state      = 42,
    verbose           = -1,
)

# Random Forest hyperparameters
RF_PARAMS = dict(
    n_estimators    = 300,
    max_features    = 0.6,
    min_samples_leaf= 10,
    random_state    = 42,
    n_jobs          = -1,
)

# Ensemble weights [lgbm, rf]
ENSEMBLE_WEIGHTS = (0.6, 0.4)

N_CV_FOLDS   = 5
ANOMALY_Z    = 2.0
RANDOM_STATE = 42

## Data Loading

In [12]:
if spark is not None:
    df_raw = spark.table(MERGED_TABLE).toPandas()
elif CSV_PATH and Path(CSV_PATH).exists():
    df_raw = pd.read_csv(CSV_PATH)
else:
    raise ValueError("Set spark + MERGED_TABLE, or a valid CSV_PATH")

df_raw.columns = [c.strip() for c in df_raw.columns]
df_raw["commodity"] = df_raw["commodity"].str.strip()
print(f"Loaded rows: {len(df_raw)}")

Loaded rows: 6823


## Preprocessing & Feature Engineering

In [13]:
def assign_region(state_abbr: str) -> str:
    s = str(state_abbr).strip().upper()
    for region, states in REGION_MAP.items():
        if s in states:
            return region
    return "Other"


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """Impute missing values and engineer features. Returns a new DataFrame."""
    df = df.copy()

    # ── Numeric cast ──────────────────────────────────────────
    numeric_cols = [
        "avg_temp_max_c", "avg_temp_min_c", "avg_temp_mean_c", "peak_temp_max_c",
        "winter_snowfall_mm", "avg_snow_depth_mm", "max_snow_depth_mm", "snow_days",
        "avg_precip_mm", "precip_std_mm", "heat_stress_days", "gdd",
        "dry_days", "heavy_rain_days", "frost_days",
        "drought_flag", "flood_flag", "extreme_heat_flag",
    ]
    for c in numeric_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # ── Imputation ────────────────────────────────────────────
    for c in ["avg_temp_max_c", "avg_temp_min_c", "avg_temp_mean_c", "peak_temp_max_c"]:
        df[c] = df[c].fillna(
            df.groupby(["commodity", "state_abbr"])[c].transform("median")
        )
    for c in ["winter_snowfall_mm", "avg_snow_depth_mm", "max_snow_depth_mm", "snow_days"]:
        df[c] = df[c].fillna(0)
    df["flood_flag"]    = df["flood_flag"].fillna(0)
    df["avg_precip_mm"] = df["avg_precip_mm"].fillna(df["avg_precip_mm"].median())
    df["precip_std_mm"] = df["precip_std_mm"].fillna(df["precip_std_mm"].median())

    # ── Feature engineering ───────────────────────────────────
    df["is_irrigated"]      = (df["irrigation"].str.strip() == "Irrigated").astype(int)
    df["temp_range"]        = df["avg_temp_max_c"] - df["avg_temp_min_c"]
    df["temp_stress_index"] = df["peak_temp_max_c"] * df["heat_stress_days"]
    df["optimal_temp_score"]= -abs(df["avg_temp_mean_c"] - 22.5)
    df["precip_cv"]         = df["precip_std_mm"] / (df["avg_precip_mm"] + 1)
    df["heat_dry_combo"]    = df["heat_stress_days"] * df["dry_days"]
    # flood included in extreme_events — previously only drought+heat, missing events like 2019 Midwest flooding
    df["extreme_events"]    = df["drought_flag"] + df["extreme_heat_flag"] + df["flood_flag"]
    df["gdd_precip"]        = df["gdd"] * df["avg_precip_mm"]
    df["gdd_per_precip"]    = df["gdd"] / (df["avg_precip_mm"] + 1)
    df["year_trend"]        = df["year"] - 2010
    yt = df["year_trend"]
    df["year_x_precip"]     = yt * df["avg_precip_mm"]
    df["year_x_heat"]       = yt * df["heat_stress_days"]

    # ── Region assignment ─────────────────────────────────────
    df["region"] = df["state_abbr"].apply(assign_region)

    return df


WEATHER_FEATURES = [
    # Temperature
    "avg_temp_max_c", "avg_temp_mean_c", "peak_temp_max_c",
    "temp_range", "temp_stress_index", "optimal_temp_score",
    # Heat stress
    "heat_stress_days", "frost_days",
    # Precipitation
    "avg_precip_mm", "dry_days", "heavy_rain_days", "precip_cv",
    # Snowfall
    "winter_snowfall_mm", "snow_days",
    # Extreme weather (drought + heat + flood all included)
    "drought_flag", "extreme_heat_flag", "flood_flag", "extreme_events", "heat_dry_combo",
    # GDD
    "gdd", "gdd_precip", "gdd_per_precip",
    # Time trend
    "year_trend", "year_x_precip", "year_x_heat",
    # Irrigation
    "is_irrigated",
]

df = build_features(df_raw)
print(f"Features ready. Shape: {df.shape}  |  Feature count: {len(WEATHER_FEATURES)}")
print(df.groupby(["commodity", "region"]).size().unstack(fill_value=0))

Features ready. Shape: (6823, 42)  |  Feature count: 26
region     Midwest  Plains  South
commodity                        
Corn          1984     927    547
Soybeans      1915     696    754


## Helper functions

In [14]:
def compute_metrics(actual, predicted):
    rmse = float(np.sqrt(mean_squared_error(actual, predicted)))
    mae  = float(mean_absolute_error(actual, predicted))
    r2   = float(r2_score(actual, predicted))
    return rmse, mae, r2


def add_fixed_effects(train: pd.DataFrame, test: pd.DataFrame):
    """Compute state and county baseline yields from train, apply to both splits.

    Baselines are the historical mean yield per state/county (train only).
    The model uses these as features alongside weather and year_trend,
    letting LightGBM learn the temporal trend directly without explicit detrending.
    """
    global_mean = train["yield_amount"].mean()

    state_mean = train.groupby("state_abbr")["yield_amount"].mean()
    train["state_baseline"] = train["state_abbr"].map(state_mean).fillna(global_mean)
    test["state_baseline"]  = test["state_abbr"].map(state_mean).fillna(global_mean)

    cmean = (
        train.groupby(["state_fips", "county_fips"])["yield_amount"]
        .mean().reset_index(name="county_baseline")
    )
    train = train.merge(cmean, on=["state_fips", "county_fips"], how="left")
    test  = test.merge(cmean, on=["state_fips", "county_fips"], how="left")
    train["county_baseline"] = train["county_baseline"].fillna(train["state_baseline"])
    test["county_baseline"]  = test["county_baseline"].fillna(test["state_baseline"])

    return train, test

## MLflow Training — Corn & Soybeans (year-based 8:2 split)

Train: years < 2019 / Test: years >= 2019

The model predicts `yield_amount` directly. Temporal trend is captured via `year_trend` feature — no explicit detrending needed. Anomaly detection flags residuals that weather features cannot explain.

In [15]:
mlflow.set_experiment(experiment_id=MLFLOW_EXPERIMENT_ID)

# Year-based train/test split — prevents time leakage
df["is_test"] = df["year"] >= TEST_START_YEAR
print(f"Train years: < {TEST_START_YEAR}  ({(~df['is_test']).sum()} rows)")
print(f"Test years : >= {TEST_START_YEAR} ({df['is_test'].sum()} rows)")

FEATURES  = WEATHER_FEATURES + ["state_baseline", "county_baseline"]
w_lgbm, w_rf = ENSEMBLE_WEIGHTS
summary   = {}
models    = {}   # (crop, region) → (lgbm_model, rf_model)
test_Xy   = {}   # (crop, region) → (X_test_df, y_test_series)
train_Xy  = {}   # (crop, region) → (X_train_df, y_train_series)

with mlflow.start_run(run_name="ensemble_yield_prediction") as parent_run:
    mlflow.log_params({
        **{f"lgbm_{k}": v for k, v in LGBM_PARAMS.items()},
        **{f"rf_{k}": v for k, v in RF_PARAMS.items()},
        "ensemble_weights": str(ENSEMBLE_WEIGHTS),
        "test_start_year": TEST_START_YEAR,
        "regions": list(REGION_MAP.keys()),
    })

    for crop in ["Corn", "Soybeans"]:
        for region in REGION_MAP.keys():
            run_name = f"ensemble_{crop.lower()}_{region.lower()}"
            with mlflow.start_run(run_name=run_name, nested=True) as child_run:
                dc    = df[(df["commodity"] == crop) & (df["region"] == region)].copy()
                train = dc[~dc["is_test"]].copy()
                test  = dc[dc["is_test"]].copy()
                print(f"\n{'='*55}\n  {crop} × {region}  Train={len(train)}, Test={len(test)}\n{'='*55}")

                if len(train) < MIN_TRAIN_ROWS:
                    print(f"  SKIP: train {len(train)} < {MIN_TRAIN_ROWS}")
                    mlflow.set_tag("skipped", "insufficient_data")
                    continue

                train, test = add_fixed_effects(train, test)

                X_train_all = train[FEATURES]
                y_train_all = train["yield_amount"]
                X_test_fin  = test[FEATURES]

                # Register MLflow datasets
                train_dataset = mlflow.data.from_pandas(
                    train[["state_fips", "county_fips", "year"] + FEATURES + ["yield_amount"]],
                    name=f"train_{crop.lower()}_{region.lower()}",
                )
                test_dataset = mlflow.data.from_pandas(
                    test[["state_fips", "county_fips", "year"] + FEATURES + ["yield_amount"]],
                    name=f"test_{crop.lower()}_{region.lower()}",
                )

                # LightGBM — early stopping on 15% hold-out
                X_tr, X_val, y_tr, y_val = train_test_split(
                    X_train_all, y_train_all, test_size=0.15, random_state=RANDOM_STATE
                )
                lgbm_model = lgb.LGBMRegressor(**LGBM_PARAMS)
                lgbm_model.fit(
                    X_tr, y_tr,
                    eval_set=[(X_val, y_val)],
                    callbacks=[lgb.early_stopping(50, verbose=False)],
                )
                print(f"  LGBM best iteration: {lgbm_model.best_iteration_}")

                # Random Forest — trained on full train set
                rf_model = RandomForestRegressor(**RF_PARAMS)
                rf_model.fit(X_train_all, y_train_all)

                # Store for post-hoc analysis (feature importance, SHAP)
                models[(crop, region)]   = (lgbm_model, rf_model)
                test_Xy[(crop, region)]  = (X_test_fin.copy(), test["yield_amount"].copy())
                train_Xy[(crop, region)] = (X_train_all.copy(), y_train_all.copy())

                # Log both models
                lgbm_info = mlflow.sklearn.log_model(
                    sk_model=lgbm_model, name=f"lgbm_{crop.lower()}_{region.lower()}",
                    input_example=X_tr.head(5),
                )
                rf_info = mlflow.sklearn.log_model(
                    sk_model=rf_model, name=f"rf_{crop.lower()}_{region.lower()}",
                    input_example=X_train_all.head(5),
                )
                logged_lgbm = mlflow.get_logged_model(lgbm_info.model_id)
                logged_rf   = mlflow.get_logged_model(rf_info.model_id)

                # Train metrics (ensemble)
                train_pred = w_lgbm * lgbm_model.predict(X_train_all) + w_rf * rf_model.predict(X_train_all)
                tr_rmse, tr_mae, tr_r2 = compute_metrics(y_train_all.values, train_pred)
                for lm, ds in [(logged_lgbm, train_dataset), (logged_rf, train_dataset)]:
                    mlflow.log_metrics(
                        metrics={"train_r2": tr_r2, "train_mae": tr_mae, "train_rmse": tr_rmse},
                        model_id=lm.model_id, dataset=ds,
                    )

                # Test metrics (ensemble)
                pred_yield   = w_lgbm * lgbm_model.predict(X_test_fin) + w_rf * rf_model.predict(X_test_fin)
                actual_yield = test["yield_amount"].values
                rmse, mae, r2 = compute_metrics(actual_yield, pred_yield)
                for lm, ds in [(logged_lgbm, test_dataset), (logged_rf, test_dataset)]:
                    mlflow.log_metrics(
                        metrics={"test_r2": r2, "test_mae": mae, "test_rmse": rmse},
                        model_id=lm.model_id, dataset=ds,
                    )

                print(f"  R2={r2:.4f}  MAE={mae:.2f}  RMSE={rmse:.2f}")
                summary[(crop, region)] = {"r2": r2, "mae": mae, "rmse": rmse}

    # Log overall summary to parent run
    for (crop, region), m in summary.items():
        key = f"{crop.lower()}_{region.lower()}"
        mlflow.log_metrics({f"{key}_r2": m["r2"], f"{key}_mae": m["mae"], f"{key}_rmse": m["rmse"]})

print("\n=== Training Summary ===")
for (crop, region), m in summary.items():
    print(f"  {crop:10s} × {region:9s}: R2={m['r2']:.4f}  MAE={m['mae']:.2f}  RMSE={m['rmse']:.2f}")

2026/06/13 14:09:21 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/13 14:09:21 INFO mlflow.store.db.utils: Updating database tables


MlflowException: No Experiment with id=2445807330421530 exists

## Feature Importance Dashboard

Aggregated feature importance across all crop × region models, weighted by training set size.

In [ ]:
import matplotlib.pyplot as plt

# Aggregate LightGBM feature importances across all models, weighted by training set size
# RF impurity-based importance is biased toward high-cardinality features (e.g. county_baseline)
# → use LightGBM importance for interpretation, RF is kept for prediction only
lgbm_imp = pd.Series(0.0, index=FEATURES)
total_n  = 0

for (crop, region), (lgbm_m, _) in models.items():
    n = len(train_Xy[(crop, region)][1])
    lgbm_imp += pd.Series(lgbm_m.feature_importances_, index=FEATURES) * n
    total_n  += n

lgbm_imp /= total_n
lgbm_imp /= lgbm_imp.sum()

# Split: weather features vs structural controls
structural = ["state_baseline", "county_baseline"]
weather_imp   = lgbm_imp.drop(structural).sort_values(ascending=True)
structural_imp = lgbm_imp[structural].sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 9))

# Left: weather features only
weather_imp.plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].set_title("Weather Feature Importance (LightGBM)\nWhat weather variables drive yield?")
axes[0].set_xlabel("Normalized Importance")
axes[0].axvline(1 / len(weather_imp), color="red", linestyle="--", alpha=0.5, label="uniform baseline")
axes[0].legend()

# Right: all features including structural controls
lgbm_imp.sort_values(ascending=True).plot(kind="barh", ax=axes[1], color="coral")
axes[1].set_title("All Features (LightGBM)\nIncluding structural controls")
axes[1].set_xlabel("Normalized Importance")

plt.suptitle("Feature Importance Dashboard — LightGBM\n"
             "(RF omitted: impurity-based importance overestimates high-cardinality features)",
             fontsize=12)
plt.tight_layout()
plt.show()

print("\nTop 10 weather features (LightGBM):")
print(weather_imp.sort_values(ascending=False).head(10).to_string())
print(f"\nStructural controls:")
print(structural_imp.sort_values(ascending=False).to_string())

## 5-Fold CV Anomaly Detection

In [ ]:
# Walk-forward CV with multi-year test windows matching the actual test period length.
# e.g., if actual test = 2021-2023 (3 years), each fold also tests on a 3-year window.
# Stride = 1 (sliding), so fold N+1 shifts cutoff by 1 year.
#
# Fold 1: train < 2014,  test = 2014-2016
# Fold 2: train < 2015,  test = 2015-2017
# Fold 3: train < 2016,  test = 2016-2018
# Fold 4: train < 2017,  test = 2017-2019
# Fold 5: train < 2018,  test = 2018-2020

test_years_actual = sorted(df[df["year"] >= TEST_START_YEAR]["year"].unique())
cv_window         = len(test_years_actual)  # match actual test duration

all_train_years   = sorted(df[df["year"] < TEST_START_YEAR]["year"].unique())
# valid cutoffs: those where a full cv_window-year test block fits inside training years
possible_cutoffs  = all_train_years[: len(all_train_years) - cv_window + 1]
cv_cutoffs        = possible_cutoffs[-N_CV_FOLDS:]  # last N_CV_FOLDS sliding positions

cv_folds = []
for cutoff in cv_cutoffs:
    idx      = all_train_years.index(cutoff)
    te_years = all_train_years[idx : idx + cv_window]
    cv_folds.append((cutoff, te_years))

print(f"Actual test years : {test_years_actual}  (window = {cv_window} years)")
print(f"CV folds ({len(cv_folds)}):")
for i, (cutoff, te_yrs) in enumerate(cv_folds):
    print(f"  Fold {i+1}: train < {cutoff},  test = {te_yrs}")

df["pred_yield"] = np.nan
w_lgbm, w_rf = ENSEMBLE_WEIGHTS

with mlflow.start_run(run_name=f"ensemble_{len(cv_folds)}fold_timecv_anomaly") as cv_run:
    mlflow.log_params({
        **{f"lgbm_{k}": v for k, v in LGBM_PARAMS.items()},
        **{f"rf_{k}": v for k, v in RF_PARAMS.items()},
        "ensemble_weights": str(ENSEMBLE_WEIGHTS),
        "n_cv_folds":          len(cv_folds),
        "anomaly_z_threshold": ANOMALY_Z,
        "regions":             list(REGION_MAP.keys()),
        "cv_type":             "walk_forward_multiyear",
        "cv_window_years":     cv_window,
    })

    fold_r2s = []

    for fold, (cutoff, te_years) in enumerate(cv_folds):
        train_mask = df["year"] < cutoff
        test_mask  = df["year"].isin(te_years)

        fold_preds, fold_actuals = [], []

        for crop in ["Corn", "Soybeans"]:
            for region in REGION_MAP.keys():
                mask = (df["commodity"] == crop) & (df["region"] == region)
                tr = df[train_mask & mask].copy()
                te = df[test_mask  & mask].copy()

                if len(tr) < MIN_TRAIN_ROWS or len(te) == 0:
                    continue

                tr, te = add_fixed_effects(tr, te)

                FEATURES = WEATHER_FEATURES + ["state_baseline", "county_baseline"]

                val_idx   = tr.sample(frac=0.15, random_state=RANDOM_STATE).index
                tr_inner  = tr.loc[~tr.index.isin(val_idx)]
                val_inner = tr.loc[val_idx]

                lgbm_model = lgb.LGBMRegressor(**LGBM_PARAMS)
                lgbm_model.fit(
                    tr_inner[FEATURES], tr_inner["yield_amount"],
                    eval_set=[(val_inner[FEATURES], val_inner["yield_amount"])],
                    callbacks=[lgb.early_stopping(50, verbose=False)],
                )

                rf_model = RandomForestRegressor(**RF_PARAMS)
                rf_model.fit(tr[FEATURES], tr["yield_amount"])

                pred_yield = (
                    w_lgbm * lgbm_model.predict(te[FEATURES]) +
                    w_rf   * rf_model.predict(te[FEATURES])
                )
                df.loc[te.index, "pred_yield"] = pred_yield

                fold_preds.extend(pred_yield.tolist())
                fold_actuals.extend(te["yield_amount"].values.tolist())

        if fold_actuals:
            fold_r2 = r2_score(fold_actuals, fold_preds)
            fold_r2s.append(fold_r2)
            mlflow.log_metric("fold_r2", fold_r2, step=fold + 1)
            print(f"  Fold {fold+1}/{len(cv_folds)} (test={te_years}) — R2={fold_r2:.4f}")

    # ── Residual → Z-score → Anomaly flag ─────────────────────
    valid_mask = df["pred_yield"].notna()
    df.loc[valid_mask, "residual"] = (
        df.loc[valid_mask, "pred_yield"] - df.loc[valid_mask, "yield_amount"]
    ).abs()

    df["residual_zscore"] = df.groupby("commodity")["residual"].transform(
        lambda x: (x - x.mean()) / x.std()
    )
    df["anomaly_flag"] = (df["residual_zscore"] > ANOMALY_Z).astype(int)

    total_anomalies = int(df["anomaly_flag"].sum())
    anomaly_pct     = float(df["anomaly_flag"].mean() * 100)
    cv_r2_mean      = float(np.mean(fold_r2s)) if fold_r2s else float("nan")

    mlflow.log_metrics({
        "cv_r2_mean":      cv_r2_mean,
        "total_anomalies": total_anomalies,
        "anomaly_pct":     anomaly_pct,
    })

    print(f"\nCV R2 mean : {cv_r2_mean:.4f}")
    print(f"Anomalies  : {total_anomalies} ({anomaly_pct:.1f}%)")

## Weather Anomaly Detection

Flags county-years where weather features themselves are statistically extreme (Z-score > 2.0).

| anomaly_type | meaning |
|---|---|
| **Extreme Weather → Yield Impact** | unusual weather AND model failed to predict → high confidence weather event |
| **Yield Mismatch (unexplained)** | model failed but weather was normal → data error or non-weather factor |
| **Extreme Weather (model handled)** | unusual weather but model predicted correctly |
| **Normal** | no flags |

In [ ]:
WEATHER_ANOMALY_Z = 2.0

# Raw weather features to monitor for extremity (interpretable ones)
WEATHER_RAW = [c for c in [
    "avg_precip_mm", "heat_stress_days", "drought_flag", "flood_flag",
    "extreme_heat_flag", "gdd", "frost_days", "dry_days", "heavy_rain_days",
    "avg_temp_mean_c", "peak_temp_max_c",
] if c in df.columns]

# Z-scores per commodity (normalize within crop type)
for feat in WEATHER_RAW:
    df[f"wz_{feat}"] = df.groupby("commodity")[feat].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-8)
    )

wz_cols = [f"wz_{f}" for f in WEATHER_RAW]
df["weather_extremity"]    = df[wz_cols].abs().max(axis=1)
df["weather_anomaly_flag"] = (df["weather_extremity"] > WEATHER_ANOMALY_Z).astype(int)

# Combined anomaly type
conditions = [
    (df["anomaly_flag"] == 1) & (df["weather_anomaly_flag"] == 1),
    (df["anomaly_flag"] == 1) & (df["weather_anomaly_flag"] == 0),
    (df["anomaly_flag"] == 0) & (df["weather_anomaly_flag"] == 1),
]
choices = [
    "Extreme Weather → Yield Impact",
    "Yield Mismatch (unexplained)",
    "Extreme Weather (model handled)",
]
df["anomaly_type"] = np.select(conditions, choices, default="Normal")

print("=== Anomaly Type Distribution ===")
print(df["anomaly_type"].value_counts().to_string())

print("\n=== Extreme Weather Events by Year ===")
extreme = df[df["weather_anomaly_flag"] == 1]
print(extreme.groupby(["year", "commodity"])["weather_anomaly_flag"].count().unstack(fill_value=0).to_string())

print("\n=== Most Extreme Weather Counties ===")
top_extreme = (
    df[df["weather_anomaly_flag"] == 1]
    .groupby(["state_abbr", "county_fips", "year"])
    .agg(weather_extremity=("weather_extremity", "max"), commodity=("commodity", "first"))
    .reset_index()
    .sort_values("weather_extremity", ascending=False)
    .head(15)
)
display(top_extreme)

## SHAP Explainer — What Drives Yield?

SHAP (SHapley Additive exPlanations) shows:
1. **Bar plot** — which features matter most on average
2. **Beeswarm** — how each feature pushes predictions up or down
3. **Waterfall** — why the model failed most on a specific county-year

In [ ]:
import shap

# Use the model with most test data for SHAP analysis
target = max(test_Xy.keys(), key=lambda k: len(test_Xy[k][1]))
target_crop, target_region = target
lgbm_m, _ = models[target]
X_test_df, y_test = test_Xy[target]

print(f"SHAP analysis: {target_crop} × {target_region}  ({len(X_test_df)} test rows)")

explainer   = shap.TreeExplainer(lgbm_m)
shap_values = explainer.shap_values(X_test_df)

# 1. Global importance (bar)
print("\n[1] Global Feature Importance — SHAP")
shap.summary_plot(shap_values, X_test_df, plot_type="bar", show=True,
                  plot_size=(10, 7))

# 2. Beeswarm — direction and magnitude per feature
print("\n[2] Feature Impact Distribution — SHAP Beeswarm")
shap.summary_plot(shap_values, X_test_df, show=True, plot_size=(10, 7))

# 3. Waterfall for the single worst-predicted row (highest residual)
pred_test   = lgbm_m.predict(X_test_df)
residuals   = np.abs(pred_test - y_test.values)
worst_pos   = int(np.argmax(residuals))
worst_info  = X_test_df.iloc[worst_pos]

print(f"\n[3] Worst Prediction Explained")
print(f"    Actual={y_test.values[worst_pos]:.1f}  Predicted={pred_test[worst_pos]:.1f}  "
      f"Residual={residuals[worst_pos]:.1f}")

shap.waterfall_plot(shap.Explanation(
    values        = shap_values[worst_pos],
    base_values   = explainer.expected_value,
    data          = worst_info.values,
    feature_names = FEATURES,
))

## Map Visualization

County-level choropleth of yield anomaly rate and extreme weather rate.
Grey counties = no corn/soybean data in the dataset.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

try:
    from pygris import counties as get_counties
    us_counties = get_counties(year=2020, cache=True)
    us_counties["state_fips"]  = us_counties["STATEFP"].astype(int)
    us_counties["county_fips"] = us_counties["COUNTYFP"].astype(int)
    us_counties = us_counties[us_counties["state_fips"] < 60]  # exclude territories

    county_agg = (
        df[df["pred_yield"].notna()]
        .groupby(["state_fips", "county_fips"])
        .agg(
            anomaly_rate         = ("anomaly_flag",         "mean"),
            weather_anomaly_rate = ("weather_anomaly_flag", "mean"),
            avg_residual_zscore  = ("residual_zscore",      "mean"),
        )
        .reset_index()
    )

    gdf = us_counties.merge(county_agg, on=["state_fips", "county_fips"], how="left")

    # --- Map 1: Yield Anomaly Rate ---
    fig, ax = plt.subplots(figsize=(24, 14))
    gdf.plot(
        column="anomaly_rate", ax=ax,
        cmap="Reds", legend=True,
        legend_kwds={
            "label": "Yield Anomaly Rate",
            "orientation": "horizontal",
            "shrink": 0.5,
            "pad": 0.02,
            "fraction": 0.03,
        },
        missing_kwds={"color": "lightgrey", "label": "No data"},
        linewidth=0.1, edgecolor="white",
    )
    ax.set_title("Yield Anomaly Rate by County\n(fraction of years with unexplained yield deviation)",
                 fontsize=18, pad=16)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    # --- Map 2: Extreme Weather Rate ---
    fig, ax = plt.subplots(figsize=(24, 14))
    gdf.plot(
        column="weather_anomaly_rate", ax=ax,
        cmap="Blues", legend=True,
        legend_kwds={
            "label": "Extreme Weather Rate",
            "orientation": "horizontal",
            "shrink": 0.5,
            "pad": 0.02,
            "fraction": 0.03,
        },
        missing_kwds={"color": "lightgrey", "label": "No data"},
        linewidth=0.1, edgecolor="white",
    )
    ax.set_title("Extreme Weather Rate by County\n(fraction of years with extreme weather conditions)",
                 fontsize=18, pad=16)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"Map rendering failed: {e}")
    print("Fallback: state-level anomaly summary")
    state_agg = (
        df[df["pred_yield"].notna()]
        .groupby("state_abbr")
        .agg(anomaly_rate=("anomaly_flag", "mean"), weather_anomaly_rate=("weather_anomaly_flag", "mean"))
        .sort_values("anomaly_rate", ascending=False)
    )
    print(state_agg.head(20).to_string())

## Anomaly Summary

## Model Dashboard

In [16]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.table import Table
import numpy as np

# ── Palette ──────────────────────────────────────────────────
C = {
    "corn":     "#E8A838",
    "soy":      "#4A90D9",
    "green":    "#27AE60",
    "red":      "#E74C3C",
    "blue":     "#2980B9",
    "bg":       "#F4F6F9",
    "card":     "#FFFFFF",
    "text":     "#2C3E50",
    "muted":    "#7F8C8D",
    "header":   "#1A252F",
}
CROP_COLOR = {"Corn": C["corn"], "Soybeans": C["soy"]}

# ── Aggregate test predictions ────────────────────────────────
w_lgbm, w_rf = ENSEMBLE_WEIGHTS
all_actual, all_pred, all_crop = [], [], []
for (crop, region), (lgbm_m, rf_m) in models.items():
    X_te, y_te = test_Xy[(crop, region)]
    p = w_lgbm * lgbm_m.predict(X_te) + w_rf * rf_m.predict(X_te)
    all_actual.extend(y_te.values.tolist())
    all_pred.extend(p.tolist())
    all_crop.extend([crop] * len(y_te))

all_actual = np.array(all_actual)
all_pred   = np.array(all_pred)
overall_r2 = float(r2_score(all_actual, all_pred))
overall_mae = float(mean_absolute_error(all_actual, all_pred))
overall_rmse = float(np.sqrt(mean_squared_error(all_actual, all_pred)))
n_train = sum(len(v[1]) for v in train_Xy.values())
n_test  = sum(len(v[1]) for v in test_Xy.values())

# ── Layout ───────────────────────────────────────────────────
fig = plt.figure(figsize=(22, 26), facecolor=C["bg"])
gs  = gridspec.GridSpec(
    4, 3, figure=fig,
    hspace=0.48, wspace=0.38,
    top=0.94, bottom=0.04, left=0.07, right=0.96,
    height_ratios=[0.8, 2.2, 2.2, 1.8],
)

fig.suptitle(
    "TerraCast  ·  Model Dashboard",
    fontsize=22, fontweight="bold", color=C["text"], y=0.975,
)
fig.text(
    0.5, 0.958,
    "Ensemble: LightGBM (60%) + Random Forest (40%)  |  Corn & Soybeans  |  Midwest / Plains / South",
    ha="center", fontsize=11, color=C["muted"],
)

def style_ax(ax, title=None):
    ax.set_facecolor(C["card"])
    for sp in ax.spines.values():
        sp.set_color("#D5D8DC"); sp.set_linewidth(0.8)
    if title:
        ax.set_title(title, fontsize=11, fontweight="bold", color=C["text"], pad=9)

# ── Row 0: KPI cards ─────────────────────────────────────────
kpis = [
    ("Test R²  (2021–2023)",        f"{overall_r2:.3f}",         C["green"] if overall_r2 >= 0.4 else C["red"]),
    ("CV R²  (walk-forward 5-fold)", f"{cv_r2_mean:.3f}",        C["corn"]),
    ("Train / Test rows",           f"{n_train:,} / {n_test:,}", C["blue"]),
]
for col, (label, value, color) in enumerate(kpis):
    ax = fig.add_subplot(gs[0, col])
    ax.set_facecolor(C["card"])
    for sp in ax.spines.values():
        sp.set_color(color); sp.set_linewidth(3)
    ax.text(0.5, 0.60, value, transform=ax.transAxes,
            ha="center", va="center", fontsize=30, fontweight="bold", color=color)
    ax.text(0.5, 0.18, label, transform=ax.transAxes,
            ha="center", va="center", fontsize=9.5, color=C["muted"])
    ax.set_xticks([]); ax.set_yticks([])

# ── Row 1 left+mid: per-model R² grouped bar ─────────────────
ax_r2 = fig.add_subplot(gs[1, :2])
style_ax(ax_r2, "Test R²  by Crop × Region  (held-out 2021–2023)")

crops_ord   = ["Corn", "Soybeans"]
regions_ord = list(REGION_MAP.keys())
x     = np.arange(len(regions_ord))
w_bar = 0.32

for ci, crop in enumerate(crops_ord):
    vals = [summary.get((crop, r), {}).get("r2", np.nan) for r in regions_ord]
    offset = (ci - 0.5) * w_bar
    bars = ax_r2.bar(x + offset, vals, w_bar,
                     color=CROP_COLOR[crop], alpha=0.88,
                     edgecolor="white", linewidth=0.6, label=crop)
    for bar, v in zip(bars, vals):
        if not np.isnan(v):
            ax_r2.text(bar.get_x() + bar.get_width()/2,
                       bar.get_height() + 0.012,
                       f"{v:.2f}", ha="center", va="bottom",
                       fontsize=9.5, fontweight="bold", color=C["text"])

ax_r2.set_xticks(x); ax_r2.set_xticklabels(regions_ord, fontsize=12)
ax_r2.set_ylim(0, 1.05); ax_r2.set_ylabel("R²", fontsize=10)
ax_r2.axhline(overall_r2, color="gray", linestyle="--",
              linewidth=1, alpha=0.7, label=f"Overall R²={overall_r2:.2f}")
ax_r2.legend(fontsize=9, loc="upper right")

# ── Row 1 right: CV fold R² ───────────────────────────────────
ax_cv = fig.add_subplot(gs[1, 2])
style_ax(ax_cv, f"Walk-Forward CV  ({len(fold_r2s)} folds)")

bar_colors = [C["blue"] if v >= cv_r2_mean else "#AED6F1" for v in fold_r2s]
bars = ax_cv.bar(range(1, len(fold_r2s)+1), fold_r2s,
                 color=bar_colors, edgecolor="white", linewidth=0.5)
ax_cv.axhline(cv_r2_mean, color=C["corn"], linestyle="--",
              linewidth=1.6, label=f"Mean={cv_r2_mean:.3f}")
ax_cv.set_ylim(0, 1.05); ax_cv.set_xlabel("Fold", fontsize=9)
ax_cv.set_ylabel("R²", fontsize=9); ax_cv.legend(fontsize=8)

for i, (bar, v, (_, te_yrs)) in enumerate(zip(bars, fold_r2s, cv_folds)):
    ax_cv.text(bar.get_x() + bar.get_width()/2,
               bar.get_height() + 0.012,
               f"{v:.3f}", ha="center", va="bottom", fontsize=8)
    ax_cv.text(i+1, -0.07, f"{te_yrs[0]}–{te_yrs[-1]}",
               ha="center", fontsize=7.5, color=C["muted"],
               transform=ax_cv.get_xaxis_transform())

# ── Row 2 left+mid: Predicted vs Actual scatter ───────────────
ax_sc = fig.add_subplot(gs[2, :2])
style_ax(ax_sc, "Predicted vs Actual Yield  (test set 2021–2023)")

for crop in ["Corn", "Soybeans"]:
    mask = np.array(all_crop) == crop
    ax_sc.scatter(all_actual[mask], all_pred[mask],
                  color=CROP_COLOR[crop], alpha=0.4, s=18, label=crop)

lo = min(all_actual.min(), all_pred.min()) - 5
hi = max(all_actual.max(), all_pred.max()) + 5
ax_sc.plot([lo, hi], [lo, hi], "k--", linewidth=1.2, alpha=0.5)
ax_sc.set_xlim(lo, hi); ax_sc.set_ylim(lo, hi)
ax_sc.set_xlabel("Actual Yield (bu/ac)", fontsize=10)
ax_sc.set_ylabel("Predicted Yield (bu/ac)", fontsize=10)
ax_sc.text(0.05, 0.92,
           f"R²={overall_r2:.3f}   MAE={overall_mae:.1f}   RMSE={overall_rmse:.1f}",
           transform=ax_sc.transAxes, fontsize=10, fontweight="bold", color=C["text"])
ax_sc.legend(fontsize=9)

# ── Row 2 right: Residual by year ────────────────────────────
ax_res = fig.add_subplot(gs[2, 2])
style_ax(ax_res, "Avg |Residual| by Year  (CV pred)")

valid = df["pred_yield"].notna()
if valid.sum() > 0:
    res_yr = (
        df[valid].assign(abs_res=(df.loc[valid,"pred_yield"] - df.loc[valid,"yield_amount"]).abs())
        .groupby("year")["abs_res"].mean()
    )
    years = res_yr.index.tolist()
    colors_yr = [C["red"] if y in [int(yr) for _, te in cv_folds for yr in te] else C["blue"]
                 for y in years]
    ax_res.bar(years, res_yr.values, color=colors_yr, alpha=0.8, edgecolor="white")
    ax_res.set_xlabel("Year", fontsize=9); ax_res.set_ylabel("|Residual| (bu/ac)", fontsize=9)
    ax_res.tick_params(axis="x", rotation=45, labelsize=8)
    legend_h = [mpatches.Patch(color=C["red"],  label="CV test year"),
                mpatches.Patch(color=C["blue"], label="CV train year")]
    ax_res.legend(handles=legend_h, fontsize=7.5)

# ── Row 3: Config table ───────────────────────────────────────
ax_tbl = fig.add_subplot(gs[3, :])
style_ax(ax_tbl, "Model Configuration")
ax_tbl.axis("off")

rows = [
    ["Models",          "LightGBM + Random Forest (ensemble)"],
    ["Ensemble weights","LightGBM 60%  +  RF 40%"],
    ["LGBM n_estimators / lr / num_leaves",
     f"{LGBM_PARAMS['n_estimators']} / {LGBM_PARAMS['learning_rate']} / {LGBM_PARAMS['num_leaves']}"],
    ["LGBM reg_alpha / reg_lambda / subsample",
     f"{LGBM_PARAMS['reg_alpha']} / {LGBM_PARAMS['reg_lambda']} / {LGBM_PARAMS['subsample']}"],
    ["RF n_estimators / max_features / min_samples_leaf",
     f"{RF_PARAMS['n_estimators']} / {RF_PARAMS['max_features']} / {RF_PARAMS['min_samples_leaf']}"],
    ["Early stopping",  "LightGBM: patience=50 on 15% holdout"],
    ["Features",        f"{len(FEATURES)} total  (weather + GDD + trend + irrigation + state/county baseline)"],
    ["Train / Test split", f"year < {TEST_START_YEAR}  /  year ≥ {TEST_START_YEAR}"],
    ["Regions",         "Midwest · Plains · South  (separate model per crop × region)"],
    ["CV strategy",     f"Walk-forward, {len(cv_folds)}-fold, {len(test_years_actual)}-year test window"],
    ["Overall MAE / RMSE", f"{overall_mae:.2f} / {overall_rmse:.2f} bu/ac  (test set)"],
    ["Anomaly detection", f"Z-score > {ANOMALY_Z}  on CV residuals  →  {int(df['anomaly_flag'].sum())} flags ({df['anomaly_flag'].mean()*100:.1f}%)"],
]

tbl = ax_tbl.table(
    cellText=rows,
    colLabels=["Parameter", "Value"],
    cellLoc="left", loc="center",
    bbox=[0.0, 0.0, 1.0, 1.0],
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9.5)
for (r, c), cell in tbl.get_celld().items():
    cell.set_edgecolor("#D5D8DC")
    if r == 0:
        cell.set_facecolor(C["header"])
        cell.set_text_props(color="white", fontweight="bold")
    elif r % 2 == 0:
        cell.set_facecolor("#EBF5FB")
    else:
        cell.set_facecolor(C["card"])
    if c == 0:
        cell.set_text_props(fontweight="bold", color=C["text"])

# ── Save ─────────────────────────────────────────────────────
out_path = "terracast_model_dashboard.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=C["bg"])
plt.show()
print(f"Saved → {out_path}")

NameError: name 'models' is not defined

In [ ]:
print("Anomaly summary by crop:")
print(df.groupby("commodity")["anomaly_flag"].agg(["sum", "mean", "count"]))

out_cols = [
    "state_fips", "county_fips", "year", "county_name", "state_abbr",
    "commodity", "irrigation", "yield_amount", "pred_yield",
    "residual", "residual_zscore", "anomaly_flag",
]
result_df = df[[c for c in out_cols if c in df.columns]].copy()

display(result_df[result_df["anomaly_flag"] == 1].sort_values("residual_zscore", ascending=False).head(20))